# MobileNetV1 — Depthwise Separable Convolutions

**Paper:** Howard, A. G., Zhu, M., Chen, B., Kalenichenko, D., Wang, W., Weyand, T., Andreetto, M., & Adam, H. (2017).  
*MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications.*  
arXiv:1704.04861. [[PDF]](https://arxiv.org/abs/1704.04861)

**Venue:** Google Inc. Technical Report, April 2017.

---

## Overview

MobileNetV1 introduced **Depthwise Separable Convolution** to make deep CNNs tractable for mobile and embedded devices. The insight is that a standard k×k convolution can be factorised into two cheaper operations:

1. **Depthwise conv** (k×k, groups=C_in): spatial filtering, one filter per input channel
2. **Pointwise conv** (1×1): linear cross-channel combination

**Cost reduction** (k=3, C_out=512): ~8.6× fewer MACs than a standard 3×3 conv.

A **width multiplier α** ∈ {0.25, 0.5, 0.75, 1.0} uniformly scales all channel counts,
giving a continuous accuracy/efficiency trade-off.

## Architecture Diagram (ASCII)

```
Input [B, 3, H, W]
    │
    ▼
┌────────────────────────────────────────────────────────┐
│  STEM                                                  │
│  Conv(3→32, 3×3, s2) + BN + ReLU → H/2                │
│  DWSConv(32→64, s1)                                    │
└────────────────────────────────────────────────────────┘
    │  [B, 64, H/2, W/2]
    ▼
┌────────────────────────────────────────────────────────┐
│  STAGE 0                                               │
│  DWSConv(64→128, s2) → DWSConv(128→128, s1)           │
└────────────────────────────────────────────────────────┘
    │  [B, 128, H/4, W/4]   ← feature_maps[0]
    ▼
┌────────────────────────────────────────────────────────┐
│  STAGE 1                                               │
│  DWSConv(128→256, s2) → DWSConv(256→256, s1)          │
└────────────────────────────────────────────────────────┘
    │  [B, 256, H/8, W/8]   ← feature_maps[1]
    ▼
┌────────────────────────────────────────────────────────┐
│  STAGE 2                                               │
│  DWSConv(256→512, s2) → 5×DWSConv(512→512, s1)        │
└────────────────────────────────────────────────────────┘
    │  [B, 512, H/16, W/16]  ← feature_maps[2]
    ▼
┌────────────────────────────────────────────────────────┐
│  STAGE 3                                               │
│  DWSConv(512→1024, s2) → DWSConv(1024→1024, s1)       │
└────────────────────────────────────────────────────────┘
    │  [B, 1024, H/32, W/32] ← feature_maps[3]
    ▼
  GlobalAvgPool → Dropout → [B, 1024] = pooled_output
```

**DWSConv block** internals:
```
  DW: Conv(C→C, 3×3, groups=C) + BN + ReLU  ← spatial filtering
  PW: Conv(C→C', 1×1)          + BN + ReLU  ← channel mixing
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

BG   = '#0f1117'
FG   = '#e8eaf6'
ACC1 = '#7986cb'  # standard conv
ACC2 = '#66bb6a'  # depthwise conv
ACC3 = '#ffa726'  # pointwise conv
ACC4 = '#ef5350'  # cost label

fig, axes = plt.subplots(1, 2, figsize=(14, 7), facecolor=BG)
fig.suptitle('MobileNetV1: Depthwise Separable Convolution', color=FG, fontsize=14, fontweight='bold')

# ── Left panel: Standard conv ─────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor(BG)
ax.set_xlim(0, 5); ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Standard 3×3 Conv', color=FG, fontsize=11)

# Input
ax.add_patch(FancyBboxPatch((1, 6.3), 3, 0.8, boxstyle='round,pad=0.1', fc=ACC1, alpha=0.8, ec=FG, lw=1))
ax.text(2.5, 6.7, 'Input [B, C_in, H, W]', ha='center', va='center', color=FG, fontsize=9)

# Conv block
ax.add_patch(FancyBboxPatch((0.5, 4.3), 4, 1.2, boxstyle='round,pad=0.1', fc=ACC1, alpha=0.5, ec=FG, lw=1.5))
ax.text(2.5, 5.0, '3×3 Conv\n(C_in → C_out)', ha='center', va='center', color=FG, fontsize=9)
ax.text(2.5, 4.45, f'Params: k² × C_in × C_out', ha='center', va='center', color=ACC4, fontsize=8)

# Arrow
ax.annotate('', xy=(2.5, 4.3), xytext=(2.5, 6.3), arrowprops=dict(arrowstyle='->', color=FG, lw=1.5))

# Output
ax.add_patch(FancyBboxPatch((1, 2.8), 3, 0.8, boxstyle='round,pad=0.1', fc=ACC1, alpha=0.8, ec=FG, lw=1))
ax.text(2.5, 3.2, 'Output [B, C_out, H, W]', ha='center', va='center', color=FG, fontsize=9)
ax.annotate('', xy=(2.5, 2.8), xytext=(2.5, 4.3), arrowprops=dict(arrowstyle='->', color=FG, lw=1.5))

# Cost for k=3, C_in=C_out=512
ax.text(2.5, 1.8, 'k=3, C_in=C_out=512:', ha='center', color=FG, fontsize=9)
ax.text(2.5, 1.2, '9 × 512 × 512 = 2,359,296 ops', ha='center', color=ACC4, fontsize=9, fontweight='bold')

# ── Right panel: Depthwise Separable ──────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor(BG)
ax2.set_xlim(0, 5); ax2.set_ylim(0, 8)
ax2.axis('off')
ax2.set_title('Depthwise Separable Conv (DWSConv)', color=FG, fontsize=11)

# Input
ax2.add_patch(FancyBboxPatch((1, 6.8), 3, 0.7, boxstyle='round,pad=0.1', fc=ACC1, alpha=0.8, ec=FG, lw=1))
ax2.text(2.5, 7.15, 'Input [B, C_in, H, W]', ha='center', va='center', color=FG, fontsize=9)

# Depthwise block
ax2.add_patch(FancyBboxPatch((0.5, 5.0), 4, 1.2, boxstyle='round,pad=0.1', fc=ACC2, alpha=0.6, ec=FG, lw=1.5))
ax2.text(2.5, 5.65, 'Depthwise 3×3 (groups=C_in)\n+ BN + ReLU', ha='center', va='center', color=FG, fontsize=9)
ax2.text(2.5, 5.05, 'Params: k² × C_in', ha='center', va='center', color=ACC4, fontsize=8)
ax2.annotate('', xy=(2.5, 5.0), xytext=(2.5, 6.8), arrowprops=dict(arrowstyle='->', color=FG, lw=1.5))

# Intermediate
ax2.add_patch(FancyBboxPatch((1, 4.2), 3, 0.5, boxstyle='round,pad=0.1', fc=ACC2, alpha=0.4, ec=FG, lw=0.8))
ax2.text(2.5, 4.45, '[B, C_in, H, W] — spatial features', ha='center', va='center', color=FG, fontsize=8)
ax2.annotate('', xy=(2.5, 4.2), xytext=(2.5, 5.0), arrowprops=dict(arrowstyle='->', color=FG, lw=1.5))

# Pointwise block
ax2.add_patch(FancyBboxPatch((0.5, 2.9), 4, 1.0, boxstyle='round,pad=0.1', fc=ACC3, alpha=0.6, ec=FG, lw=1.5))
ax2.text(2.5, 3.5, 'Pointwise 1×1\n+ BN + ReLU', ha='center', va='center', color=FG, fontsize=9)
ax2.text(2.5, 2.95, 'Params: C_in × C_out', ha='center', va='center', color=ACC4, fontsize=8)
ax2.annotate('', xy=(2.5, 2.9), xytext=(2.5, 4.2), arrowprops=dict(arrowstyle='->', color=FG, lw=1.5))

# Output
ax2.add_patch(FancyBboxPatch((1, 2.2), 3, 0.5, boxstyle='round,pad=0.1', fc=ACC1, alpha=0.8, ec=FG, lw=1))
ax2.text(2.5, 2.45, 'Output [B, C_out, H, W]', ha='center', va='center', color=FG, fontsize=9)
ax2.annotate('', xy=(2.5, 2.2), xytext=(2.5, 2.9), arrowprops=dict(arrowstyle='->', color=FG, lw=1.5))

# Cost
ax2.text(2.5, 1.5, 'k=3, C_in=C_out=512:', ha='center', color=FG, fontsize=9)
ax2.text(2.5, 0.9, '9×512 + 512×512 = 267,264 ops', ha='center', color=ACC4, fontsize=9, fontweight='bold')
ax2.text(2.5, 0.3, '→ 8.8× cheaper!', ha='center', color='#66bb6a', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('mobilenet_v1_dws_diagram.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()
print('Figure saved.')

## Key Innovations & Mathematics

### Depthwise Separable Convolution

**Standard 3×3 Conv** computes:
$$\mathbf{G}_{k,l,n} = \sum_{i,j,m} \mathbf{K}_{i,j,m,n} \cdot \mathbf{F}_{k+i-1, l+j-1, m}$$

Computational cost: $D_K^2 \cdot M \cdot N \cdot D_F^2$ (where $D_K$=kernel size, $M$=input channels, $N$=output channels, $D_F$=feature map size)

**Depthwise Separable** factorises this into:

1. **Depthwise** (spatial):
$$\hat{\mathbf{G}}_{k,l,m} = \sum_{i,j} \hat{\mathbf{K}}_{i,j,m} \cdot \mathbf{F}_{k+i-1, l+j-1, m}$$
Cost: $D_K^2 \cdot M \cdot D_F^2$

2. **Pointwise** (channel mixing):
$$\mathbf{G}_{k,l,n} = \sum_m \mathbf{W}_{m,n} \cdot \hat{\mathbf{G}}_{k,l,m}$$
Cost: $M \cdot N \cdot D_F^2$

**Reduction ratio:**
$$\frac{D_K^2 \cdot M \cdot D_F^2 + M \cdot N \cdot D_F^2}{D_K^2 \cdot M \cdot N \cdot D_F^2} = \frac{1}{N} + \frac{1}{D_K^2}$$

For $D_K=3, N=512$: reduction ≈ $\frac{1}{512} + \frac{1}{9} \approx 0.113$ → **~8.8× cheaper**.

### Width Multiplier α

All channel counts scaled by $\alpha \in \{0.25, 0.5, 0.75, 1.0\}$:
$$\text{cost}(\alpha) \approx \alpha^2 \cdot \text{cost}(1.0)$$

| Model | α | MACs | Top-1 Acc (ImageNet) |
|-------|---|------|---------------------|
| V1-100 | 1.0 | 569M | 70.6% |
| V1-075 | 0.75 | 325M | 68.4% |
| V1-050 | 0.5 | 149M | 63.7% |
| V1-025 | 0.25 | 41M  | 50.6% |

In [ ]:
# ── Code Walkthrough ──────────────────────────────────────────────────────────
import sys; sys.path.insert(0, '../../src')

import torch
from hale_core.nn.vision.cnn.mobilenet_v1 import _DWSConv, MobileNetV1

# 1. Inspect the DWSConv building block
dws = _DWSConv(in_chs=64, out_chs=128, stride=2)
print('DWSConv(64→128, s=2):')
print(dws)

x = torch.randn(1, 64, 32, 32)
y = dws(x)
print(f'\nInput shape:  {tuple(x.shape)}')
print(f'Output shape: {tuple(y.shape)}')

# 2. Depthwise params vs. standard
std_params = 3*3*64*128
dw_params  = 3*3*64          # depthwise: one filter per input channel
pw_params  = 64*128          # pointwise: cross-channel 1x1
print(f'\nStandard 3×3 params: {std_params:,}')
print(f'DW params:           {dw_params:,}')
print(f'PW params:           {pw_params:,}')
print(f'DW+PW params:        {dw_params+pw_params:,} ({(dw_params+pw_params)/std_params:.3f}× of standard)')

In [ ]:
# ── Usage Example ─────────────────────────────────────────────────────────────
from hale_core.nn.vision import build_vision_backbone

# Build MobileNetV1 with full width
backbone = build_vision_backbone('mobilenet_v1_100', img_size=224)
backbone.eval()

x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out = backbone(x)

print('MobileNetV1-1.0 outputs:')
print(f'  pooled_output: {tuple(out.pooled_output.shape)}')
for i, fm in enumerate(out.feature_maps):
    print(f'  feature_maps[{i}]: {tuple(fm.shape)}')

print(f'\nParameters: {backbone.num_parameters():,}')

# Compare variants
for name in ['mobilenet_v1_025', 'mobilenet_v1_050', 'mobilenet_v1_075', 'mobilenet_v1_100']:
    b = build_vision_backbone(name)
    print(f'  {name}: embed_dim={b.embed_dim}, params={b.num_parameters():,}')

## References

| # | Citation |
|---|----------|
| [1] | Howard, A. G., Zhu, M., Chen, B., Kalenichenko, D., Wang, W., Weyand, T., Andreetto, M., & Adam, H. (2017). **MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications.** arXiv:1704.04861. |
| [2] | Sandler, M., Howard, A., Zhu, M., Zhmoginov, A., & Chen, L.-C. (2018). **MobileNetV2: Inverted Residuals and Linear Bottlenecks.** CVPR. arXiv:1801.04381. |
| [3] | Howard, A., Sandler, M., Chu, G., et al. (2019). **Searching for MobileNetV3.** ICCV. arXiv:1905.02244. |
| [4] | He, K., Zhang, X., Ren, S., & Sun, J. (2016). **Deep Residual Learning for Image Recognition.** CVPR. |

---
*Notebook: MobileNetV1 — HaleBlocks backbone family*  
*Architecture: `hale_core.nn.vision.cnn.mobilenet_v1`*